__Paper__: Using neural networks as an alternative to air dispersion modeling in environmental impact assessment.

__Authors__: Mateo Concha and Gonzalo A. Ruz.

__Description__: Code used for the paper development.

In [7]:
import pandas as pd
import numpy as np

# --- Load datasets (adjust paths if needed) ---
train_df     = pd.read_csv("train.csv")
val_df       = pd.read_csv("validation.csv")
testF_df     = pd.read_csv("test_FFEE.csv")
testCV_df    = pd.read_csv("test_CV.csv")

target_ffee = "FFEE"
target_cv   = "Chacaya"

# --- Split y ---
y_train = train_df[target_ffee].values
y_val   = val_df[target_ffee].values
y_test_FFEE = testF_df[target_ffee].values
y_test_CV   = testCV_df[target_cv].values

# --- Build X with consistent feature columns
feature_cols = [c for c in train_df.columns if c != target_ffee]

X_train_raw = train_df[feature_cols].values
X_val_raw   = val_df[feature_cols].values
X_test_FFEE_raw = testF_df[feature_cols].values

# For CV test, drop its target and then reindex to training feature order
X_test_CV_raw = testCV_df.drop(columns=[target_cv]).reindex(columns=feature_cols).values

print("Raw shapes:",
      "train", X_train_raw.shape,
      "val", X_val_raw.shape,
      "test_FFEE", X_test_FFEE_raw.shape,
      "test_CV", X_test_CV_raw.shape)


Raw shapes: train (6000, 24) val (1000, 24) test_FFEE (1750, 24) test_CV (2947, 24)


In [6]:
import numpy as np
from sklearn.svm import SVR
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import mean_squared_error, mean_absolute_error
from scipy.stats import loguniform

RANDOM_STATE = 123

def rmse(y_true, y_pred):
    return float(np.sqrt(mean_squared_error(y_true, y_pred)))

def report(name, y_true, y_pred):
    print(f"{name}: RMSE={rmse(y_true, y_pred):.3f} | MAE={mean_absolute_error(y_true, y_pred):.3f}")


# Pipeline: StandardScaler + SVR(RBF)
pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("svr", SVR(kernel="rbf"))
])

param_distributions = {
    "svr__C": loguniform(1e-1, 1e3),
    "svr__gamma": loguniform(1e-4, 1e1),
    "svr__epsilon": loguniform(1e-3, 5e-1),
}

search = RandomizedSearchCV(
    estimator=pipe,
    param_distributions=param_distributions,
    n_iter=80,
    scoring="neg_root_mean_squared_error",
    cv=5,
    random_state=RANDOM_STATE,
    n_jobs=-1,
    verbose=1
)

search.fit(X_train_raw, y_train)
print("Best CV params:", search.best_params_)
print("Best CV RMSE:", -search.best_score_)

# Report on fixed validation split
best_model = search.best_estimator_
val_pred = best_model.predict(X_val_raw)
report("Validation (fixed split)", y_val, val_pred)

# Optional: refit on train+val for final test reporting
X_tr_all = np.vstack([X_train_raw, X_val_raw])
y_tr_all = np.concatenate([y_train, y_val])

final_model = search.best_estimator_
final_model.fit(X_tr_all, y_tr_all)

pred_test_f = final_model.predict(X_test_FFEE_raw)
pred_test_cv = final_model.predict(X_test_CV_raw)

report("Test FFEE", y_test_FFEE, pred_test_f)
report("Test CV", y_test_CV, pred_test_cv)


Fitting 5 folds for each of 80 candidates, totalling 400 fits
Best CV params: {'svr__C': 17.736728364798434, 'svr__epsilon': 0.002137625756452097, 'svr__gamma': 0.0010162428590517456}
Best CV RMSE: 15.294642041138639
Validation (fixed split): RMSE=8.852 | MAE=6.708
Test FFEE: RMSE=6.320 | MAE=4.228
Test CV: RMSE=5.528 | MAE=4.245
